In [ ]:
import numpy as np
import pandas as pd

import spikeinterface as si # core only
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.exporters as sexp

from probeinterface import get_probe
import probeinterface.plotting as prb_plotting
import spikeinterface.widgets as sw

In [ ]:
''' File paths '''
# path to .rhd intan file
intan_folder = "path/to/your/intan/outputs/"

# path to various info files
map_file_path = 'Z:/Isabel/ephys/SILICON PROBE MAP H10_spikesort.xlsx'

# output folder for kilosort
ks_folder = f"{intan_folder}kilosort_si/"

In [ ]:
''' Load the recording file '''
# raw neural data
recording = se.read_intan(f"{intan_folder}info.rhd", stream_id='0')

In [ ]:
# my hacky way of making sure the channels are in order...you could modify
def map_contacts_to_intan(probe, map_file_path):
    '''
    Given probe contact positions and a mapping file,
    get the index mapping each probe contact to an Intan channel

    probe_pos : ndarray, shape (n_contacts, 2)
        XY positions of the probe contacts in microns
    '''
    # load the channel map
    ch_map = pd.read_excel(map_file_path)
    
    # get the map channel positions
    xpos = ch_map["xpos"].to_numpy()
    ypos = ch_map["ypos"].to_numpy()
    map_pos = np.column_stack([xpos, ypos])
    if np.max(map_pos) < 1: # units are in mm
        map_pos = map_pos*1000
    n_channels = map_pos.shape[0]

    # get the probe contact positions
    probe_pos = probe.contact_positions
    
    # match using nearest neighbors
    tree = cKDTree(map_pos)
    distances, map_idx = tree.query(probe_pos, k=1)
    assert np.max(distances) < 0.1
    assert len(map_idx) == probe.get_contact_count()

    # map intan channel index to probe contact
    intan_ch_idx = ch_map["Intan channel"].to_numpy()
    intan_ch_idx = intan_ch_idx[map_idx]
    contact_sort = np.argsort(intan_ch_idx)
    
    # get custom channel names
    ch_names_unsorted = []
    shank_idx_unsorted = np.zeros(n_channels)
    for j, i in enumerate(map_idx):
        shank_id = str(ch_map["Shank Letter"][i])
        shank_row = str(ch_map["Shank Row"][i])
        shank_col = str(ch_map["Shank Column"][i])
        if len(shank_row) == 1:
            ch_names_unsorted.append(f"{shank_id}-0{shank_row}-{shank_col}")
        else:
            ch_names_unsorted.append(f"{shank_id}-{shank_row}-{shank_col}")
        if map_pos[i, 0] > 100:
            shank_idx_unsorted[j] = 1

    # reorder
    ch_names = []
    for i in contact_sort:
        ch_names.append(ch_names_unsorted[i])
    shank_idx = shank_idx_unsorted[contact_sort]

    return contact_sort, ch_names, shank_idx

In [ ]:
''' Add the probe map and check channel layout '''
# load the probe layout
probe = get_probe(manufacturer="cambridgeneurotech", probe_name="ASSY-236-H10")

# map to intan
contact_sort, ch_names, shank_idx = map_contacts_to_intan(probe, map_file_path)
probe = probe.get_slice(contact_sort)

# visualize to check
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
prb_plotting.plot_probe(probe, ax=ax)
ylims = ax.get_ylim()
ax.set_ylim(ylims[0], 400)
for pos, ch_name in zip(probe.contact_positions, ch_names):
    ax.text(pos[0], pos[1], ch_name)

In [ ]:
# set the channel indices in the probe object
n_channels = contact_sort.shape[0]
probe.set_device_channel_indices(np.arange(n_channels))

# add the probe map
recording = recording.set_probe(probe)
recording.set_property("channel_name", ch_names)
recording.set_property("group", shank_idx)

In [ ]:
''' Optionally remove broken channels'''
# check for noisy channels
noise_level = si.get_noise_levels(spre.common_reference(spre.highpass_filter(recording)))
plt.plot(noise_level) # to set noise thresh

In [ ]:
# set noise threshold and remove noisy channels
noise_thresh = 16
keep_ch_idx = noise_level < noise_thresh
recording = recording.select_channels(recording.channel_ids[keep_ch_idx])

In [ ]:
# sanity check and channel count
print(recording.get_num_channels())
print(recording.get_property("group").shape)
print(recording.get_property("channel_name").shape)
print(recording.get_channel_locations().shape)

In [ ]:
# reset the probe device indices
probe = recording.get_probe()
probe.set_device_channel_indices(
    np.arange(probe.get_contact_count())
)

# visualize to check
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
prb_plotting.plot_probe(probe, ax=ax)
ylims = ax.get_ylim()
ax.set_ylim(ylims[0], 400)
for pos, ch_name in zip(probe.contact_positions, recording.get_property("channel_name")):
    ax.text(pos[0], pos[1], ch_name)

In [ ]:
recording = recording.set_probe(probe)
assert (recording.get_num_channels() == recording.get_probe().get_contact_count())

In [ ]:
''' Save as a binary file for kilosort '''
rec_bin = recording.save(format="binary", folder=f"{intan_folder}recording_binary")

In [ ]:
''' Sort the data '''
# this should save everything in the ks_folder and then you can run phy as normal
sort_ks4 = ss.run_sorter(sorter_name="kilosort4", 
                              recording=rec_bin, 
                              folder=ks_folder, 
                              torch_device='cuda', 
                              skip_kilosort_preprocessing=False,
                              nblocks=0,
                              verbose=True)